In [6]:
from torch.utils.data import Dataset 
from PIL import Image
import os 

class CellDataset(Dataset): 
    def __init__(
        self, 
        cell_types: list[str], 
        augmentation_factor: int = 1,  
        set_type: str = "train", 
        transform=None
    ): 
        self.cell_types = cell_types 
        self.set_type = set_type 
        self.images: list[tuple] = []
        self.transform = transform 
        self.augmentation_factor = augmentation_factor

        for cell_type in self.cell_types: 
            cell_path: str = os.path.join("..", self.set_type, cell_type, "01") 
            segment_gt_path: str = os.path.join("..", self.set_type, cell_type, "01_GT", "SEG")

            for image_name in os.listdir(segment_gt_path): 
                cell_segment_image_path: str = os.path.join(segment_gt_path, image_name)
                image_number: str = image_name.split("man_seg")[1].split(".tif")[0]
                cell_image_path: str = os.path.join(cell_path, f"t{image_number}.tif")

                self.images.append((
                    cell_type, 
                    cell_image_path,
                    cell_segment_image_path
                ))
    
    def __len__(self): 
        return len(self.images) * self.augmentation_factor
    
    def __getitem__(self, index: int): 
        real_index = index % len(self.images)
        cell_type, cell_image_path, cell_segment_image_path = self.images[real_index]
        cell_image = Image.open(cell_image_path).convert("L")
        cell_segment_image = Image.open(cell_segment_image_path).convert("L")

        if self.transform:
            cell_image, cell_segment_image = self.transform(cell_image, cell_segment_image)

        return cell_type, cell_image, cell_segment_image
                 
    

In [ ]:
from torchvision.transforms import v2 
augmentation_factor = 3

probs = 1 - (1 - (augmentation_factor - 1) 
         / augmentation_factor) ** 0.25 

transform = v2.Compose([
    v2.RandomApply(
        transforms=[
            v2.ElasticTransform(
                alpha=90, 
                sigma=8, 
            )
        ], 
        p = probs
    ), 

    v2.RandomApply(
        transforms = [
            v2.RandomAffine(
                degrees=10, 
                translate=(0.1, 0.1), 
                scale=(0.9, 1.1), 
                shear=10
            ), 
        ], 
        p = probs, 
    ), 
    
    v2.RandomApply(
        transforms = [
            v2.ColorJitter(
                brightness=0.2, 
                contrast=0.2,
            )
        ], p = probs 
    ), 

    v2.RandomApply(
        transforms = [
            v2.RandomRotation(20)
        ], p = probs 
    )
])

0.24016431434840746
